# Thermal conductivity of fin material

The thermal conductivity is determined according to EN 13445-3:2021 Annex O

Here the thermel conductivity is given as a polynomial of the form 

$$
\lambda(T) = c_0 + c_1 T + c_2 T ^2 
$$

The coefficients $c_0,c_1,c_2$ are given on a per material group basis, material groups are given by "CEN ISO/TR 15608:2013, Welding —(ISO/TR 15608:2013).
Guidelines for a metallic material grouping system" but can also be seen https://ped-online.com/iso-tr-15608-groups/

For the non alloyed carbon steels from EN 10216-2:2024 such as P235GH and P265GH they belong in material group 1.1 and their coefficients are given as
$$
\begin{align*}
c_0 = 55.72 && c_1 = -2.464 \text{ E-2} && -1.298 \text{ E-5} 
\end{align*} 
$$
as per table O.4 in EN 13445-3:2021

# Fin efficiency

The fin efficency is given as 

$$
\eta_f = \frac{\tanh(X)}{X}
$$

Where 
$$
X = \varphi \frac{d_o}{2} \sqrt{\frac{2 h}{\lambda_f \delta}}
$$

Where $d_o$ is the outer diameter, $\lambda_f$ is the thermal conductivity, $\delta$ is the fin thickness and $h$ is ther heat transfer coefficient. 

For annular fins the parameter $\varphi$ is given as 
$$
\varphi_{annular} = \left( \frac{D}{d_o}-1  \right) (1+ 0.35 \ln\left( \frac{D}{d_o} \right) )
$$

For square fins it's given as 
$$
\begin{align*}
    \varphi' = 1.28 \frac{b_f}{d_o} \sqrt{\frac{l_f}{b_f}-0.2} \\
    \varphi = (\varphi'-1)(1+0.35\ln(\varphi'))
\end{align*} 
$$


In [2]:
# Import libraries
import numpy as np
from functools import cached_property
from enum import StrEnum

In [3]:
# Import enums
class finType(StrEnum):
    '''Enum for fin types'''
    Rectangular = "rectangular"
    Circular    = "circular"
    Spiral      = "spiral"

In [ ]:
# import geometry class
class Geometry:
    """Class representing the geometry of a heat exchanger with defined dimensions."""

    class _Tube:
        """Inner class representing tube geometry."""

        def __init__(self,parent):
            self.parent = parent
            self.geom_set = False

        def set_geometry(self, outer_diameter, wall_thickness):
            """Set the geometric properties of the tube.

            Args:
                outer_diameter (float): Outer diameter of the tube in meters.
                wall_thickness (float): Wall thickness of the tube in meters.

            Notes:
                Derived properties such as inner diameter, perimeters, and
                internal area are computed automatically.
            """
            self.geom_set       = True
            self.outer_diameter = outer_diameter  
            self.wall_thickness = wall_thickness 

            # Derived properties
            self.inner_diameter     = self.outer_diameter - 2 * self.wall_thickness
            self.inner_perimeter    = np.pi * self.inner_diameter
            self.outer_perimeter    = np.pi * self.outer_diameter
            self.inner_area         = np.pi * (self.inner_diameter/2)**2

    class _Duct:
        """Inner class representing duct geometry."""

        def __init__(self,parent):
            self.parent = parent
            self.geom_set = False

        def set_geometry(self, a, b):
            """Set the geometric properties of the duct.

            Args:
                a (float): Duct height in meters.
                b (float): Duct width in meters.

            Notes:
                The frontal cross-sectional area is computed automatically.
            """
            self.geom_set = True
            self.a  = a
            self.b  = b

            # Derived properties
            self.area = self.a * self.b

    class _Fin:
        """Inner class representing fin geometry."""

        def __init__(self,parent):
            self.parent = parent
            self.geom_set = False

        def set_geometry(self, average_fin_thickness, fin_height, fin_spacing, fin_type=finType.Rectangular):
            """Set the geometric properties of the fins.

            Args:
                average_fin_thickness (float): Average fin thickness in meters.
                fin_height (float): Height of the fin measured from tube OD in meters.
                fin_spacing (float): Spacing between fins (fin pitch) in meters.
                fin_type (finType): Type of fin geometry. Defaults to finType.Rectangular.

            Raises:
                NotImplementedError: If the fin type is Spiral.

            Notes:
                Additional derived geometric parameters are computed depending
                on the selected fin type.
            """
            self.geom_set = True
            self.average_fin_thickness  = average_fin_thickness
            self.fin_height             = fin_height
            self.fin_spacing            = fin_spacing
            self.fin_type               = fin_type
        
        @cached_property
        def square_height(self):
            """Compute the square fin height.

            Returns:
                float: Tube outer diameter plus twice the fin height.
            Raises:
                ValueError: If the fin type is not Rectangular.
                ValueError: If the tube geometry is not set before computing square height.

            """
            if not self.fin_type == finType.Rectangular:
                raise ValueError("Square height is only defined for rectangular fins.")
            if not self.parent.Tube.geom_set:
                raise ValueError("Tube geometry must be set before computing square height.")
            
            return self.parent.Tube.outer_diameter+2*self.fin_height
        
        
        @cached_property
        def finning_diameter(self):
            """Compute the finning diameter.

            Returns:
                float: Tube outer diameter plus twice the fin height.
            Raises:
                ValueError: If the fin type is not Circular.
                ValueError: If the tube geometry is not set before computing finning diameter.

            """
            if not self.fin_type == finType.Circular:
                raise ValueError("Finning diameter is only defined for circular fins.")
            if not self.parent.Tube.geom_set:
                raise ValueError("Tube geometry must be set before computing finning diameter.")
            
            return self.parent.Tube.outer_diameter+2*self.fin_height
        
        @cached_property
        def Psi_r(self):
            """Compute the fin efficiency coefficient for finned tubes.

            Returns:
                float: Fin coefficient Psi_r.
                
            Raises:
                ValueError: If the tube geometry is not set before computing Psi_r.

            Notes:
                Assumes constant fin thickness for simplification.
            """
            
            if not self.parent.Tube.geom_set:
                raise ValueError("Tube geometry must be set before computing Psi_r.")
            
            match self.fin_type:
                case finType.Rectangular:
                    Psi_r = (2*(self.square_height**2 - 0.785 * self.parent.Tube.outer_diameter**2
                            + 2 * self.square_height*self.average_fin_thickness)) \
                            / (np.pi * self.parent.Tube.outer_diameter * self.fin_spacing) \
                            + (1 - self.average_fin_thickness/self.fin_spacing)
                case finType.Circular: 
                    Psi_r = 1/(2*self.parent.Tube.outer_diameter*self.fin_spacing) * \
                            (self.finning_diameter**2 - self.parent.Tube.outer_diameter**2
                            + 2*self.finning_diameter * self.average_fin_thickness) \
                            + (1 - self.average_fin_thickness/self.fin_spacing)
            return Psi_r

    class _Bank:
        """Inner class representing bank geometry."""

        def __init__(self,parent):
            self.parent = parent
            self.geom_set = False

        def set_geometry(self, transverse_number_of_rows, longitudinal_number_of_rows,
                         transverse_pitch, longitudinal_pitch, L_ccrs,finned_tube_section, arrangement):
            """Set the geometric configuration of the tube bank.

            Args:
                transverse_number_of_rows (int): Number of tube rows transverse to flow.
                longitudinal_number_of_rows (int): Number of tube rows along flow direction.
                transverse_pitch (float): Transverse pitch between tubes in meters.
                longitudinal_pitch (float): Longitudinal pitch between tubes in meters.
                L_ccrs (float): Tube length at the cross-section.
                finned_tube_section (float): Length of the tube covered by fins (m).
                arrangement (arrangementType): Tube arrangement type.

            Notes:
                Derived properties such as diagonal pitch and total tube count
                are computed automatically.
            """
            self.geom_set = True

            # Row numbers
            self.transverse_number_of_rows      = transverse_number_of_rows
            self.longitudinal_number_of_rows    = longitudinal_number_of_rows
            self.total_number_of_tubes          = self.transverse_number_of_rows * self.longitudinal_number_of_rows

            # Pitches
            self.transverse_pitch               = transverse_pitch
            self.longitudinal_pitch             = longitudinal_pitch
            self.diagonal_pitch                 = np.sqrt((1/4)*self.transverse_pitch**2 + self.longitudinal_pitch**2)

            # Lengths
            self.L_ccrs                         = L_ccrs
            self.finned_tube_segment           = finned_tube_section
            
            # Arrangement
            self.arrangement                    = arrangement 

        @cached_property
        def phi_parameter(self):
            """Compute the phi parameter used in flow area calculations.

            Returns:
                float: Ratio involving transverse and diagonal pitch minus
                the conventional diameter.
            """
            if not self.parent.Tube.geom_set or not self.parent.Fin.geom_set:
                raise ValueError("Tube and Fin geometries must be set before computing phi parameter.")
            
            d_cl = self.parent.Conventional_diameter
            phi_cl = (self.transverse_pitch-d_cl)/(self.diagonal_pitch-d_cl)
            return phi_cl

    def __init__(self):
        # Initialize inner classes
        self.Tube = self._Tube(self)
        self.Duct = self._Duct(self)
        self.Fin  = self._Fin(self)
        self.Bank = self._Bank(self)

    @cached_property
    def Conventional_diameter(self):
        """Compute the conventional finned-tube diameter.

        Returns:
            float: Conventional diameter based on fin geometry.
        """
        d_cl = self.Tube.outer_diameter + \
        ((2*self.Fin.fin_height*self.Fin.average_fin_thickness)/(self.Fin.fin_spacing))
        return d_cl

    @cached_property
    def Free_flow_area(self):
        """Compute the free-flow area available for air passage.

        Returns:
            float: Free-flow area in square meters.

        Notes:
            The expression depends on the phi parameter and conventional diameter.
        """
        d_cl = self.Conventional_diameter
        phi_cl = self.phi_parameter

        if phi_cl <= 2:
            F = self.Duct.a*self.Duct.b - self.Bank.transverse_number_of_rows*self.Bank.L_ccrs*d_cl
        else:
            F = (self.Duct.a*self.Duct.b - self.Bank.transverse_number_of_rows*self.Bank.L_ccrs*d_cl)*(2/phi_cl)
        return F

    @cached_property
    def Equivalent_diameter(self):
        """Compute the equivalent hydraulic diameter for the finned geometry.

        Returns:
            float: Equivalent diameter in meters.

        Notes:
            Adjustments are applied depending on the phi parameter.
        """
        phi_cl = self.phi_parameter
        
        d_eq = (2*(self.Fin.fin_spacing*(self.Bank.transverse_pitch-self.Tube.outer_diameter)
                - 2*self.Fin.fin_height*self.Fin.average_fin_thickness)) / \
               (2*self.Fin.fin_height + self.Fin.fin_spacing)
        
        if phi_cl > 2:
            d_eq = (2*d_eq)/phi_cl

        return d_eq

    @cached_property
    def A_total_over_F(self):
        """Compute the ratio of total heat-transfer area to free-flow area.

        Returns:
            float: Ratio A_total / F.
        """
        return (np.pi*(self.Tube.outer_diameter*self.Fin.fin_spacing
                + 2*self.Fin.fin_height*self.Fin.average_fin_thickness
                + 2*self.Fin.fin_height*(self.Fin.fin_height+self.Tube.outer_diameter))) / \
               (self.Bank.transverse_pitch*self.Fin.fin_spacing
                - (self.Tube.outer_diameter*self.Fin.fin_spacing
                + 2*self.Fin.fin_height*self.Fin.average_fin_thickness))

    @cached_property
    def S_1_over_S_2(self):
        """Compute the ratio of transverse pitch to longitudinal pitch.

        Returns:
            float: S1 / S2.
        """
        return self.Bank.transverse_pitch/self.Bank.longitudinal_pitch

    @cached_property
    def sigma_1(self):
        """Compute the transverse pitch-to-diameter ratio.

        Returns:
            float: Transverse pitch divided by tube outer diameter.
        """
        return self.Bank.transverse_pitch / self.Tube.outer_diameter
    
    @cached_property
    def sigma_2(self):
        """Compute the longitudinal pitch-to-diameter ratio.

        Returns:
            float: Longitudinal pitch divided by tube outer diameter.
        """
        return self.Bank.longitudinal_pitch / self.Tube.outer_diameter

    @cached_property
    def A_r(self):
        """Compute the total fin heat-transfer area.

        Returns:
            float: Fin surface area in square meters.

        Notes:
            The expression depends on fin type and tube bank configuration.
        """
        match self.Fin.fin_type:
            case finType.Rectangular:
                A_r = 2 * (self.Fin.square_height**2 - 0.785 * self.Tube.outer_diameter**2
                        + 2 * self.Fin.square_height*self.Fin.average_fin_thickness) \
                        * self.Bank.finned_tube_segment / self.Fin.fin_spacing \
                        * self.Bank.total_number_of_tubes
            case finType.Spiral | finType.Circular:
                A_r = np.pi/2 * (self.Fin.finning_diameter**2 - self.Tube.outer_diameter**2
                        + 2 * self.Fin.finning_diameter * self.Fin.average_fin_thickness) \
                        * self.Bank.finned_tube_segment / self.Fin.fin_spacing \
                        * self.Bank.total_number_of_tubes
        return A_r
        
    @cached_property
    def A_t(self):
        """Compute the tube heat-transfer area. Excluding the area covered by fins.
        
        Returns:
            float: Tube surface area in square meters.
        """
        L_covered_by_fins = self.Bank.finned_tube_segment * (self.Fin.average_fin_thickness/self.Fin.fin_spacing)
        L_bare_tube = (self.Bank.L_ccrs - L_covered_by_fins)* self.Bank.total_number_of_tubes
        
        A_t = np.pi * self.Tube.outer_diameter * L_bare_tube
        return A_t
    
    @cached_property
    def A(self):
        """Compute the total heat-transfer area.

        Returns:
            float: Sum of fin area and tube area.
        """
        return self.A_r + self.A_t
    



In [41]:

geom = Geometry()
geom.Tube.set_geometry(31.8e-3, 4e-3)
geom.Fin.set_geometry(2e-3,35e-3 - 31.8e-3,9e-3, finType.Rectangular)
geom.Duct.set_geometry(3,3)
geom.Bank.set_geometry(10,10, 50e-3, 50e-3, 3,3-2*100e-3, "staggered")

from scipy.special import iv, kn

# Calculation of fin efficiency based on VDI section M.1
def fin_efficiency_VDI(T_base,h,geom):
    lambda_fin = lambda T: 55.72 -2.464e-2 * T - 1.298e-5 * T**2
    lambda_guess = lambda_fin(T_base)
    
    type = geom.Fin.fin_type
    
    match type:
        # Rectangular fin efficiency calculation:
        case finType.Rectangular:
            # Assuming perfect square fin and not rectangular fin
            phi_star = 1.28 * geom.Fin.square_height/geom.Tube.outer_diameter * np.sqrt(0.8)
            phi = (phi_star-1)*(1+0.35*np.log(phi_star)) 
        case finType.Circular:
            phi = (geom.Fin.finning_diameter/geom.Tube.outer_diameter)*(1+0.35*np.log(geom.Fin.finning_diameter/geom.Tube.outer_diameter)) 
    
    X = phi * geom.Tube.outer_diameter/2 * np.sqrt((2*h)/(lambda_guess * geom.Fin.average_fin_thickness) )
    
    eta_fin = np.tanh(X)/X
    
    return eta_fin

def fin_efficiency_incopera(T_base,h,geom):
    lambda_fin = lambda T: 55.72 -2.464e-2 * T - 1.298e-5 * T**2
    lambda_guess = lambda_fin(T_base)
    
    type = geom.Fin.fin_type
    
    r_1 = geom.Tube.outer_diameter/2
    r_2 = geom.Fin.finning_diameter/2
    
    m = np.sqrt((2*h)/(lambda_guess * geom.Fin.average_fin_thickness))
    
    C = (2*r_1)/ ( m* (r_2**2 - r_1**2) )
    
    K_0 = lambda r: kn(0,m*r)
    K_1 = lambda r: kn(1,m*r)
    I_0 = lambda r: iv(0,m*r)
    I_1 = lambda r: iv(1,m*r)
    eta_fin = C * (K_1(r_1)*I_1(r_2)- I_1(r_1)*K_1(r_2) )/(K_0(r_1)*I_1(r_2) + I_0(r_1)*K_1(r_2))
    
    return eta_fin 

def iterative_temperature_calculation(func,h,T_base,T_gas,geom):
    T_surface = T_base
    error = 1
    tol = 1e-6
    max_iter = 100
    iter_count = 0
    
    while error > tol and iter_count < max_iter:
        eta = func(T_surface,h,geom)
        T_new = T_base - (eta * (T_base - T_gas)) 
        error = abs(T_new - T_surface)
        T_surface = T_new
        iter_count += 1
    
    if iter_count == max_iter:
        print("Warning: Maximum iterations reached without convergence.")
    
    return T_surface, eta

def heat_transfer_rate(func,h,T_base,T_gas,geom):
    
    eta = func(T_base,h,geom)
    
    if geom.Fin.fin_type == finType.Rectangular:
        A_fin = geom.Fin.square_height **2 - np.pi*(geom.Tube.outer_diameter/2)**2
    elif geom.Fin.fin_type == finType.Circular:
        A_fin = (geom.Fin.finning_diameter**2 - geom.Tube.outer_diameter**2)/4 * np.pi
        
    return eta * h * A_fin * (T_base - T_gas)

print(heat_transfer_rate(fin_efficiency_VDI, 40, 70, 25, geom))
geom.Fin.fin_type = finType.Circular
print(heat_transfer_rate(fin_efficiency_VDI, 40, 70, 25, geom))
print(heat_transfer_rate(fin_efficiency_incopera, 40, 70, 25, geom))

print("---")
print(fin_efficiency_VDI(70,40,geom))
print(iterative_temperature_calculation(fin_efficiency_VDI,40,70,25,geom))


lambda_fin = lambda T: 55.72 -2.464e-2 * T - 1.298e-5 * T**2
lambda_fin(70)


1.1841747296313803
0.5757098269880792
0.6315922874990308
---
0.9089986582513154
(np.float64(29.024422126540898), np.float64(0.9105683971879801))


53.931597999999994